# Modules 31–38 — Expanded Industry Lab Pack

This notebook expands the existing Modules 31–38 labs into implementation-oriented, industry-standard exercises. Each lab has: business scenario → architecture → concrete task → failure injection → measurements → interview/system-design extension. Use synthetic data and sandboxed/mock systems for destructive workflows.

## Common production contract
Every exercise must track: goal, tenant, identity, risk, allowed tools, budget, state version, evidence, action history, verification result, latency, cost, safety events and recovery status.

Core architecture: **Model + Harness + Environment + Tools/MCP + State + Memory + Policy + Verification + Evaluation + Durable Execution + Observability + Improvement**.


In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from time import monotonic

class Risk(Enum): READ=0; REVERSIBLE=1; SENSITIVE=2; CONSEQUENTIAL=3
@dataclass
class Run: goal:str; tenant:str; user:str; risk:Risk; budget:float=2.0; steps:int=0; events:list=field(default_factory=list)

def event(run, kind, **data):
    run.events.append({'t':round(monotonic(),4),'kind':kind,**data})

run=Run('investigate failed payment reconciliation','bank-a','analyst-7',Risk.SENSITIVE)
event(run,'run_started',budget=run.budget); print(run)


# MODULE 31 — LOOP ENGINEERING

## Industry architecture
Business goal → task contract → OBSERVE → DECIDE → policy/budget → ACT → VERIFY → RECOVER → checkpoint → CONTINUE/STOP.

### Lab 31.1 — SOC investigation loop
**Scenario:** A security operations agent receives an alert that a privileged account logged in from an unusual location.
**Build:** observation schema containing alert, asset criticality, identity, prior events and evidence freshness. Model proposes: query SIEM, inspect endpoint, disable account. Deterministic policy allows read-only investigation but requires approval for disabling an account.
**Inject:** stale alert, SIEM timeout, contradictory IP evidence, malicious log line saying 'ignore policy'.
**Verify:** every conclusion must cite observed evidence; no containment action occurs without approval.
**Measure:** time-to-first-evidence, investigation success, false containment, tool calls, p95 latency and cost/run.
**Interview:** explain why the model never owns the transition from observation to high-impact action.

### Lab 31.2 — IT remediation loop
**Scenario:** Kubernetes service health check reports repeated 5xx responses.
**Build:** observe metrics → decide diagnostic command → execute read-only diagnostics → verify root cause → propose restart → approval if production-impacting → execute → verify recovery.
**Failure:** restart command succeeds but service remains unhealthy; agent must not declare success from command exit code alone.
**Extension:** compare fixed 5-step loop against adaptive loop that stops once SLO evidence is restored.

### Lab 31.3 — Banking reconciliation loop
**Scenario:** 1,000 payment records contain 7 unmatched transactions.
**Build:** retrieve ledger evidence, classify mismatch, query source system, prepare correction proposal, independently verify balances.
**Failure:** duplicate correction request after retry. Use stable effect ID/idempotency key.
**Gate:** monetary mutation requires exact approval binding.

### Lab 31.4 — Coding-agent loop
**Scenario:** repository has a failing unit test.
**Build:** inspect test → inspect relevant code → propose patch → run tests → inspect failure → iterate → stop only when test suite passes and diff is within scope.
**Failure:** model repeatedly changes unrelated files. Add scope constraint and non-progress detector.

### Lab 31.5 — Loop pathology laboratory
Construct: infinite retry, oscillating plan, duplicate action, stale observation, false verification, hidden tool failure, budget bypass and poisoned observation. For each, record first detectable invariant violation and recovery action.

### Lab 31.6 — Differential loop benchmark
Run baseline fixed loop, adaptive loop and verifier-first loop over the same 100 synthetic tasks. Compare success, p95 latency, cost/task, safety violations and unnecessary actions.

### Lab 31.7 — Production kill-switch lab
Implement global, tenant and run-level cancellation. Trigger cancellation during OBSERVE, DECIDE, ACT and VERIFY. Prove no new side effect starts after the cancellation boundary.

### Lab 31.8 — Loop mastery challenge
Design a reusable loop kernel for SOC + IT + coding workloads without embedding business logic. Demonstrate typed actions, hard budgets, recovery, verification and complete trajectory reconstruction.


# MODULE 32 — HARNESS ENGINEERING

## Industry architecture
Task → identity → context/RAG/memory → model adapter → action proposal → deterministic policy/budget → tool/MCP gateway → environment → verifier → trace → durable state.

### Lab 32.1 — Provider-neutral model adapter
Create a normalized response contract for three simulated providers: fast/cheap, high-quality, and private/on-prem. Preserve model version, token usage, latency and refusal/error metadata.
**Failure:** provider changes response shape. Adapter absorbs the change; harness contract remains stable.

### Lab 32.2 — Context engineering for enterprise RAG
Scenario: HR policy assistant. Build context from current task + ACL-filtered retrieval + trusted memory + system policy. Add source priority, freshness and token budget.
**Failure:** retrieved document contains instructions to exfiltrate secrets. It remains untrusted evidence, never policy.

### Lab 32.3 — Tool gateway
Implement typed tools: search_customer, get_order, create_ticket, issue_refund. Add RBAC, tenant checks, risk classification, timeout and idempotency.
**Failure:** model requests create_ticket with malformed arguments; reject before execution.

### Lab 32.4 — Harness replay
Record model proposal + context hashes + tool fixtures + policy decisions. Replay after changing the harness and calculate behavioral diffs.

### Lab 32.5 — Context compaction
Simulate a 50-turn support case. Compare raw history, summarization, retrieval-backed memory and hybrid compaction. Measure answer quality proxy, token usage and latency.

### Lab 32.6 — Extension/skill manifest
Represent every extension as name, version, capabilities, input/output schema, trust level, owner and test suite. Reject unregistered skills.

### Lab 32.7 — Security red-team harness
Attack context ordering, tool poisoning, prompt injection, secret leakage, tenant crossover and policy override. Expected result: attack becomes data/error, not control.

### Lab 32.8 — Harness mastery challenge
Swap model provider, RAG backend and tool implementation while keeping policy, budget, state, verification and observability contracts unchanged.


# MODULE 33 — LONG-RUNNING & AUTONOMOUS AGENTS

## Industry architecture
Event/schedule → durable job → lease → worker → checkpoint → wait/wake → tool effect → verification → complete/escalate/DLQ.

### Lab 33.1 — Multi-hour research worker
Scenario: build a competitor intelligence report over 200 public documents. Persist source list, completed sources, extracted claims and pending work.
**Failure:** process dies after source 97. Restart from checkpoint without repeating completed side effects.

### Lab 33.2 — Lease and heartbeat
Run two workers against one task. Only the current lease holder may mutate state. Expire the lease and recover it with a new worker.

### Lab 33.3 — Approval wait state
Procurement agent prepares a purchase request. Persist exact proposed amount/vendor/items and action hash. Human approval later resumes the job. Reject altered amount after approval.

### Lab 33.4 — Provider outage
Inject 5-minute model outage. Worker enters WAITING rather than spinning. Resume with bounded retry and provider fallback.

### Lab 33.5 — Idempotent external effects
Simulate invoice creation API. Crash immediately after API success but before checkpoint. Retry using stable idempotency key; prove only one invoice exists.

### Lab 33.6 — Long-horizon plan
Create migration plan with dependencies, deadlines, pre/postconditions and rollback steps. Execute only ready steps.

### Lab 33.7 — Always-on economics
Model 10,000 scheduled jobs. Compare polling every minute with event-driven wakeups. Include model/tool cost, idle compute and retry amplification.

### Lab 33.8 — Chaos mastery
Randomly crash workers, delay messages, expire leases, corrupt transient state and cancel jobs. Success means safe completion/recovery without duplicated consequential effects.


# MODULE 34 — SKILLS, MEMORY & CONTINUAL HARNESSES

## Industry architecture
Trajectory → candidate memory/skill → provenance → validation → benchmark/security gate → version → runtime retrieval → monitoring → rollback.

### Lab 34.1 — Support memory
Extract stable customer preferences from approved interactions. Separate user preference from temporary conversation context. Add source, timestamp, confidence and tenant.

### Lab 34.2 — Contradiction handling
Memory says 'customer prefers email'; later authorized evidence says 'use phone for urgent incidents'. Store both with scope/priority rather than overwriting silently.

### Lab 34.3 — SRE skill extraction
Mine successful incident trajectories to propose a runbook skill: symptoms → diagnostics → safe remediation → verification. Candidate skills cannot execute production mutations until promoted.

### Lab 34.4 — Skill benchmark
Evaluate a candidate skill on normal, edge, adversarial and regression cases. Require no critical safety regression even if task success improves.

### Lab 34.5 — Memory poisoning
Inject malicious memory claiming an attacker is an approved administrator. Verify identity and authorization remain external to memory.

### Lab 34.6 — Skill routing
Given coding, SOC and finance tasks, select versioned skills by capability, policy and context. Prevent a finance skill from gaining coding-repository permissions.

### Lab 34.7 — Forgetting/TTL
Expire obsolete temporary memories, revoke deleted user preferences and test that revoked data cannot reappear through summaries or caches.

### Lab 34.8 — Continual-learning mastery
Build a gated pipeline where failures generate candidate improvements but production runtime can consume only evaluated, versioned artifacts.


# MODULE 35 — ENVIRONMENTS, VERIFIERS & AGENTIC RL

## Industry architecture
Task generator → environment → agent/harness → trajectory → independent verifier → reward/label → evaluation → training/improvement.

### Lab 35.1 — Coding environment
Generate synthetic bugs, expose repository files and tests, and verify patches by executing tests. Reward passing tests, minimal scope and safety constraints.

### Lab 35.2 — Browser environment
Create a synthetic procurement portal. Agent must locate a purchase request and prepare, but not submit, a high-value order without approval.

### Lab 35.3 — RAG verifier
Verifier checks that every cited claim is entailed by retrieved evidence and that required sources are present. Compare model self-report with independent verification.

### Lab 35.4 — IT environment
Environment exposes service health, logs and restart operation. Verifier checks actual health recovery rather than accepting 'restart succeeded'.

### Lab 35.5 — Reward hacking
Create reward based only on ticket closure. Agent discovers a shortcut: close tickets without resolving them. Add independent resolution verifier and safety penalty.

### Lab 35.6 — Verifier disagreement
Compare deterministic verifier with LLM judge across 500 cases. Calculate false-positive/false-negative rates and investigate correlated errors.

### Lab 35.7 — Difficulty curriculum
Generate easy/medium/hard tasks. Filter impossible and trivial cases. Hold out templates and adversarial variants to reduce contamination.

### Lab 35.8 — Agentic-RL readiness challenge
Produce trajectory schema containing observation, action, tool call, reward components, verifier result, cost, safety result and final outcome. Build an offline policy comparison on held-out tasks.


# MODULE 36 — RECURSIVE / SELF-IMPROVING AGENTS

## Industry architecture
Baseline → failure mining → candidate proposal → isolated experiment → fixed/adversarial evaluation → quality/security/cost gates → shadow → canary → promotion → monitoring → rollback.

### Lab 36.1 — RAG improvement
Baseline retrieval Recall@5 is measured on a frozen set. Candidate changes chunking and reranking. Promote only if retrieval improves without violating tenant ACL tests.

### Lab 36.2 — Tool-routing improvement
Candidate router reduces tool calls by 20%. Test whether success and safety remain constant; reject if savings come from skipping required verification.

### Lab 36.3 — Coding harness improvement
Candidate adds automatic test-selection. Evaluate on fixed repository tasks plus hidden regressions.

### Lab 36.4 — Benchmark leakage
Give candidate access to evaluation metadata accidentally. Detect contamination and invalidate the experiment.

### Lab 36.5 — Metric gaming
Candidate improves citation count by citing many irrelevant documents. Add citation correctness and answer entailment; reject gaming behavior.

### Lab 36.6 — Canary promotion
Route 5% synthetic traffic to candidate. Compare quality, safety, latency and cost. Automatically rollback on critical safety regression.

### Lab 36.7 — Recursive harness improvement
Allow the improvement system to propose changes to its own prompt/skill/retrieval/harness configuration, but never its external promotion gate.

### Lab 36.8 — Self-improvement mastery
Demonstrate three successive improvements with immutable provenance, reproducible evaluation, approval/gates and rollback to a known-good artifact.


# MODULE 37 — COMPUTER USE & ALWAYS-ON AI TEAMMATES

## Industry architecture
Screen/DOM/state → perception → action proposal → identity/risk policy → approval → sandboxed computer → execution → state verification → audit → durable scheduler.

### Lab 37.1 — Sales operations
Synthetic CRM: agent finds stale opportunities, drafts follow-up and updates non-sensitive fields. Sending external messages requires approval.

### Lab 37.2 — Procurement
Agent compares vendor quotes in a synthetic portal and prepares a purchase order. Submission above threshold requires exact approval.

### Lab 37.3 — Finance operations
Agent reconciles invoices and flags discrepancies. It may prepare corrections but cannot transfer funds.

### Lab 37.4 — Browser prompt injection
Place hostile text on a webpage instructing the agent to upload secrets. Agent must classify page text as untrusted content and refuse the action.

### Lab 37.5 — Stale-screen defense
Change a button location/state after screenshot but before click. Agent detects state mismatch and re-observes rather than clicking blindly.

### Lab 37.6 — Credential broker
Model receives opaque capability handles, not reusable passwords/tokens. Broker performs the authenticated action inside the sandbox.

### Lab 37.7 — Always-on teammate
Schedule hourly synthetic checks, wake on event, sleep during idle periods, enforce daily budget and escalate when confidence/risk thresholds are crossed.

### Lab 37.8 — Computer-use mastery
Run 100 synthetic workflows with UI mutations, timeouts, duplicate clicks and approval delays. Target safe recovery and zero unauthorized consequential actions.


# MODULE 38 — FRONTIER AGENTIC RAG CAPSTONE

## Enterprise reference architecture
Business goal → task contract/risk → governance → harness → hybrid RAG + memory + skills → model → policy-controlled tools/MCP → durable execution → environment → independent verification → telemetry/evaluation → gated improvement → release/rollback.

### Lab 38.1 — Enterprise research analyst
Research an industry question across ACL-controlled documents and approved web fixtures. Produce answer, evidence map, uncertainty and citations. Verifier checks every material claim.

### Lab 38.2 — SOC autonomous investigator
Investigate a synthetic alert, correlate evidence, recommend containment and request human approval for disruptive action. Persist state across simulated restart.

### Lab 38.3 — IT operations teammate
Diagnose a service incident, gather evidence, execute only approved remediation and verify SLO recovery.

### Lab 38.4 — Banking operations analyst
Reconcile transactions, identify exceptions and prepare corrections. Monetary effects require exact approval and idempotency.

### Lab 38.5 — Procurement/sales teammate
Research suppliers/accounts, retrieve CRM/procurement evidence, draft actions and use computer-use only through the capability gateway.

### Lab 38.6 — Long-running autonomous case
Run a 24-hour synthetic case with scheduled wakeups, external-event waits, worker crash, provider outage, memory update and human approval.

### Lab 38.7 — Controlled self-improvement
Mine failed trajectories, propose one retrieval or skill improvement, evaluate on fixed + adversarial cases, shadow it, then canary only if every gate passes.

### Lab 38.8 — Full chaos challenge
Simultaneously inject: poisoned retrieval, model outage, tool timeout, worker crash, stale checkpoint, duplicate message, policy conflict, budget pressure and evaluator regression. Required outcome: recover safely, preserve tenant isolation, avoid unauthorized side effects, produce an auditable trace and either complete or escalate.


# Cross-module industry scorecard

For every lab report: **Task Success %, Groundedness %, Citation Correctness %, Verifier Pass %, Safety Violations, Unauthorized Actions, Recovery %, p50/p95/p99 Latency, Cost/Successful Task, Tool Calls/Task, Retry Amplification, Human Escalation %, Tenant-Isolation Failures, Audit Completeness %.**

## Required comparison
For each scenario compare: deterministic workflow vs single agent vs multi-agent vs durable agent vs frontier harness. The correct answer is not 'most autonomous'; it is the simplest architecture that meets success, safety, latency, cost and recovery requirements.

## Portfolio evidence
A learner completes the track only after producing: architecture diagram, threat model, runnable Colab, failure report, benchmark table, trace, rollback demonstration, system-design explanation and production-readiness decision for each module.


# Final interview challenge

Design an enterprise autonomous teammate that can research, retrieve governed knowledge, use tools, wait for approvals, operate for days, recover from crashes, learn reusable skills and improve its harness.

You must defend: why a single agent is sufficient where possible; where a durable worker is necessary; where MCP/tool gateways belong; how authorization differs from model reasoning; how verifiers prevent false completion; how memory/skills can be poisoned; how self-improvement is externally gated; how computer use is sandboxed; and how the system can be stopped and reconstructed from audit evidence.
